# Focused circos: CENP-A + 195/389-bp + 349-bp tandem repeats

New pycirclize track layout, derived from `pyCircularize.ipynb`.

Changes vs. the original notebook:
- **Removed** the interspersed-repeat density track and the (broad) tandem-repeat density track.
- **Added** tracks for the **195-bp + 389-bp centromeric repeats** (combined into a single track).
- **Added** CENP-A CUT&Tag signal plotted *as a bin* (mean signal per fixed window), **on a log10 scale**.
- Kept the **349-bp pericentromeric repeat** track (innermost).

Radial order, **outside → inside**:
1. **Chromosomes** (r = 95-100) — tick marks + scaffold junctions
2. **CENP-A** (r = 94-86) — mean signal per bin on a log10(signal+1) scale, drawn as bars
3. **195-bp + 389-bp centromeric repeats** (r = 85-76) — TRF tandem repeats, one combined track
4. **349-bp pericentromeric repeats** (r = 75-66)

Run the cells top to bottom. The CENP-A binning step caches its output to a TSV,
so re-runs after the first are fast.

In [8]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pycirclize import Circos
from Bio.SeqFeature import SeqFeature, FeatureLocation
import pyBigWig

# Render figures inline
%matplotlib inline

In [ ]:
# ============================================================================
# Paths & parameters
# ============================================================================

# Final assembly index (chr sizes)
ASSEMBLY_FAI = ("/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/"
                "data/denovo_OctDegus_genome/041425-assembly/"
                "hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/"
                "assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai")

# TRF tandem-repeat tables
TRF_DIR = ("/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/"
           "code/command-line-script/genome-annotation/trf-tandem-repeat")
REP349 = f"{TRF_DIR}/349peak_repeat_1millionBpMin.tsv"   # 349-bp centromeric repeats
REP195 = f"{TRF_DIR}/195peak_repeat.tsv"                 # 195-bp tandem repeats
REP389 = f"{TRF_DIR}/389peak_repeat.tsv"                 # 389-bp tandem repeats

# Scaffold junction coordinates (contig boundaries within chromosomes)
JUNCTION_BED = "../feature-overview/agp_final_contig2scaffold.bed"

# CENP-A CUT&Tag bigWig (XG_150 = CENP-A rep1; XG_151 = CENP-A rep2)
CENPA_BW = ("/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/"
            "figure/cenpa-cuttag-centromere/bw_files/XG_150.all.bw")

# CENP-A binning resolution (bp). Smaller = finer bins / more bars.
CENPA_BIN_SIZE = 500_000
# Cache file for the binned CENP-A table (written on first run)
CENPA_CACHE = "assembly_final.sorted.headerRenamed.CENPA.binned.tsv"
print("paths OK")

In [ ]:
# ============================================================================
# 1) Chromosome sizes (karyotype)
# ============================================================================
fai = pd.read_csv(
    ASSEMBLY_FAI,
    sep="\t", header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"],
)
karyotype = fai[fai["chr"].str.contains("chr")][["chr", "length"]].reset_index(drop=True)
chr_sizes = dict(zip(karyotype["chr"], karyotype["length"]))

print(f"{len(karyotype)} chromosomes: {', '.join(karyotype['chr'])}")
karyotype.head()

In [ ]:
# ============================================================================
# 2) Load TRF tandem-repeat tables (349 / 195 / 389 bp)
# ============================================================================
# Each table has one row per repeat region with columns incl.
# sequence, start, end, chromosome, match_percent, repeat_point_relative_perc, ...
df_349 = pd.read_csv(REP349, sep="\t", index_col=False)
df_195 = pd.read_csv(REP195, sep="\t", index_col=False)
df_389 = pd.read_csv(REP389, sep="\t", index_col=False)

print("349-bp repeats:", df_349.shape)
print("195-bp repeats:", df_195.shape)
print("389-bp repeats:", df_389.shape)
df_195[["chromosome", "start", "end"]].head()

In [ ]:
# ============================================================================
# 3) Scaffold junction coordinates (contig boundaries within each chromosome)
# ============================================================================
junction_bed = pd.read_csv(
    JUNCTION_BED,
    sep="\t", header=0,
    dtype={"chr": "string", "start": "int64", "end": "int64"},
    names=["chr", "start", "end"],
    na_values=["."],
    keep_default_na=False,
)
print("junctions:", junction_bed.shape)
junction_bed.head()

In [ ]:
# ============================================================================
# 4) CENP-A CUT&Tag signal, binned into fixed windows
#    (read from bigWig with pyBigWig; cached to TSV on first run)
# ============================================================================
def bin_bigwig_signal(bw_path, chromosomes, bin_size):
    """Return a DataFrame (Chromosome, Start, End, signal) of mean bigWig
    signal per fixed-size bin, for the given chromosomes."""
    bw = pyBigWig.open(bw_path)
    rows = []
    for chrom in chromosomes:
        size = bw.chroms().get(chrom)
        if size is None:
            continue
        for start in range(0, size, bin_size):
            end = min(start + bin_size, size)
            v = bw.stats(chrom, start, end, type="mean")
            val = v[0] if v and v[0] is not None else 0.0
            rows.append((chrom, start, end, val))
    bw.close()
    return pd.DataFrame(rows, columns=["Chromosome", "Start", "End", "signal"])

if os.path.exists(CENPA_CACHE):
    df_cenpa = pd.read_csv(CENPA_CACHE, sep="\t")
    print(f"Loaded CENP-A bins from cache: {CENPA_CACHE}")
else:
    df_cenpa = bin_bigwig_signal(CENPA_BW, karyotype["chr"].tolist(), CENPA_BIN_SIZE)
    df_cenpa.to_csv(CENPA_CACHE, sep="\t", index=False)
    print(f"Computed CENP-A bins ({len(df_cenpa)} bins) -> saved {CENPA_CACHE}")

# Log10 transform of the signal: log10(signal + 1) keeps zero-signal bins at 0
df_cenpa["signal_log10"] = np.log10(df_cenpa["signal"].clip(lower=0) + 1)

print("total bins:", df_cenpa.shape)
df_cenpa.head()

In [ ]:
# ============================================================================
# 5) Circos plot
#    Radial order (outermost -> innermost):
#      1) Chromosomes                  r=95-100   ticks + scaffold junctions
#      2) CENP-A (bins, log10)         r=94-86    mean CENP-A signal per bin
#      3) 195-bp + 389-bp centromeric  r=85-76    TRF tandem repeats (combined)
#      4) 349-bp pericentromeric       r=75-66    TRF tandem repeats
# ============================================================================
chr_sizes = dict(zip(karyotype["chr"], karyotype["length"]))
circos = Circos(sectors=chr_sizes, start=30, end=359, space=2, endspace=False)

# Robust scale for the log bins: cap at the 99.5th percentile so one extreme
# outlier (e.g. a multimapping pile-up) saturates instead of flattening the track.
cenpa_vmax_raw = float(np.percentile(df_cenpa["signal"], 99.5))
cenpa_vmax_log = float(np.log10(cenpa_vmax_raw + 1))

for i, sector in enumerate(circos.sectors):

    # ------------------------------------------------------------------
    # 2) CENP-A track as binned bars (right inside the chromosome track)
    #    Bar heights are log10(signal+1), clipped to the robust cap;
    #    y-axis ticks show raw signal values.
    # ------------------------------------------------------------------
    cenpa_track = sector.add_track((94, 86), r_pad_ratio=0.05)
    cenpa_track.axis(fc="none", ec="grey", lw=0.5)
    sub_cenpa = df_cenpa[df_cenpa["Chromosome"] == sector.name]
    if not sub_cenpa.empty:
        cenpa_track.bar(
            x=sub_cenpa["Start"].values,
            height=np.clip(sub_cenpa["signal_log10"].values, 0, cenpa_vmax_log),
            width=(sub_cenpa["End"] - sub_cenpa["Start"]).values,
            align="edge",
            vmin=0, vmax=cenpa_vmax_log,
            color="darkred", ec="none", alpha=0.85,
        )

    # ----------------------------------------------------------------
    # 3) 195-bp + 389-bp centromeric repeats, combined into one track
    # ----------------------------------------------------------------
    trf_track = sector.add_track((85, 76), r_pad_ratio=0.05)
    trf_track.axis(fc="none", ec="grey", lw=0.5)
    feat_195 = [
        SeqFeature(FeatureLocation(int(r.start), int(r.end)), type="repeat")
        for _, r in df_195[df_195["chromosome"] == sector.name].iterrows()
    ]
    if feat_195:
        trf_track.genomic_features(feat_195, fc="#F8766D", ec="#F8766D", lw=0.3, alpha=0.8)
    feat_389 = [
        SeqFeature(FeatureLocation(int(r.start), int(r.end)), type="repeat")
        for _, r in df_389[df_389["chromosome"] == sector.name].iterrows()
    ]
    if feat_389:
        trf_track.genomic_features(feat_389, fc="#00BFC4", ec="#00BFC4", lw=0.3, alpha=0.8)

    # -----------------------------------------------------------------
    # 4) 349-bp pericentromeric repeats (innermost track)
    # -----------------------------------------------------------------
    cent_track = sector.add_track((75, 66), r_pad_ratio=0.05)
    cent_track.axis(fc="none", ec="grey", lw=0.5)
    feat_349 = [
        SeqFeature(
            FeatureLocation(int(r.start), int(r.end)),
            type="repeat",
            qualifiers={
                "match_percent": r.match_percent,
                "repeat_point_relative_perc": r.repeat_point_relative_perc,
            },
        )
        for _, r in df_349[df_349["chromosome"] == sector.name].iterrows()
    ]
    if feat_349:
        cent_track.genomic_features(feat_349, fc="gold", ec="goldenrod", lw=0.5, alpha=0.8)

    # -----------------------------------------------------------------
    # 1) Chromosome track: scaffold junctions + major/minor ticks
    # -----------------------------------------------------------------
    junction_track = sector.add_track((95, 100))
    chr_junctions = junction_bed[junction_bed["chr"] == sector.name]
    for _, row in chr_junctions.iterrows():
        junction_track.line(
            x=[row["start"], row["end"]],
            y=[0, 100],
            color="black", lw=1, arc=True, ls=":",
        )

    outer = sector.add_track((95, 100))
    outer.axis(fc="none", ec="black", lw=0.5)
    outer.xticks_by_interval(
        interval=50_000_000, tick_length=3, outer=True, show_label=True,
        label_size=8, label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        line_kws=dict(ec="grey"),
    )
    outer.xticks_by_interval(
        interval=10_000_000, tick_length=1, outer=True, show_label=False,
        line_kws=dict(ec="black", lw=0.5),
    )

    mid = (sector.start + sector.end) / 2
    sector.text(text=sector.name, x=mid, r=115, adjust_rotation=True, size=10)

    # Y-axis ticks on the first sector, labelled with raw signal values
    if i == 0:
        tick_raw = [v for v in (1, 5, 10, 25, 50) if v <= cenpa_vmax_raw]
        if not tick_raw:
            tick_raw = [round(cenpa_vmax_raw, 1)]
        tick_pos = [float(np.log10(v + 1)) for v in tick_raw]
        cenpa_track.yticks(
            y=tick_pos,
            labels=[f"{v:g}" for v in tick_raw],
            side="left", tick_length=2, label_size=7,
        )

# ============================================================================
# Render + track labels (in the 0-deg gap) + legend for 195 vs 389
# ============================================================================
fig = circos.plotfig()
ax = fig.axes[0]

track_labels = [
    (94, "1) Chromosomes", "black"),
    (86, "2) CENP-A (log10)", "darkred"),
    (76, "3) 195-bp + 389-bp\n    centromeric repeats", "black"),
    (66, "4) 349-bp pericentromeric\n    repeats", "black"),
]
for track_top, text, color in track_labels:
    ax.text(
        np.deg2rad(0), track_top + 1.5, text, rotation=0,
        ha="left", va="bottom", color=color, size=8, fontweight="bold",
    )

from matplotlib.patches import Patch
handles = [
    Patch(fc="#F8766D", ec="none", alpha=0.8, label="195-bp repeats"),
    Patch(fc="#00BFC4", ec="none", alpha=0.8, label="389-bp repeats"),
]
ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(-0.12, 1.12),
          frameon=True, fontsize=8)

fig.tight_layout()
fig.show()

In [ ]:
# ============================================================================
# 6) Save figure
# ============================================================================
fig.savefig("degus_genome_circos_CENPA_195_389_349.png", dpi=300, bbox_inches="tight")
fig.savefig("degus_genome_circos_CENPA_195_389_349.svg", bbox_inches="tight")
print("Saved: degus_genome_circos_CENPA_195_389_349.png / .svg")